In [1]:
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
html_db_path = "../../data/2025-2026/html_db/ncsl_db_2025_2026.html"

with open(html_db_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

html_content = soup.find_all('div', id='dnn_ctr34167_StateNetDB_linkList')[0]

In [3]:
def parse_statenet_html(html_snippet):
    soup = html_snippet

    # Each bill block is separated by <hr/> tags or contained within the main linkList div.
    # We can split the containers or find all blocks by looking at the main link structures.
    # Looking at your snippet, each bill starts with a <div> containing a bill link.

    records = []

    # Find all main bill link tags to anchor each bill entry
    bill_links = soup.find_all(
        'a', href=lambda href: href and 'resources.cgi?id=ID:bill:' in href
    )

    for link in bill_links:
        rec = {}

        # 1. Bill Name & Year (contained right around the first <a> tag)
        bill_text_parts = (
            [t.strip() for t in link.next_sibling.strings if t.strip()]
            if link.next_sibling
            else []
        )
        rec['Bill'] = link.get_text(strip=True)

        # We need to find the parent container or traverse siblings until the next bill/hr
        # Let's gather all siblings until the next <hr> or end of elements
        current = link.parent
        block_elements = []
        while current:
            if current.name == 'hr':
                break
            block_elements.append(current)
            current = current.next_sibling

        # Combine text or parse fields from this specific block
        block_soup = BeautifulSoup(
            ''.join([str(e) for e in block_elements]), 'html.parser')

        # 2. Stage / Sub-header (the bold element right after the year div, if present)
        stage_div = block_soup.find(
            'div', style=lambda val: val and 'font-weight: bold' in val
        )
        if stage_div:
            rec['Stage'] = stage_div.get_text(strip=True)

        # 3. Standard labeled fields (Status, Date of Last Action, Author, Topics, Summary)
        for b_tag in block_soup.find_all('b'):
            b_text = b_tag.get_text(strip=True).replace(':', '')
            # Get the text immediately following the <b> tag until the next <br> or element
            val_text = ''
            sibling = b_tag.next_sibling
            while (
                sibling
                and sibling.name != 'br'
                and not getattr(sibling, 'name', None) == 'b'
            ):
                val_text += str(sibling)
                sibling = sibling.next_sibling

            val_clean = BeautifulSoup(
                val_text, 'html.parser').get_text(strip=True)

            if 'Status' in b_text:
                rec['Status'] = val_clean
            elif 'Date of Last Action' in b_text:
                rec['Date of Last Action'] = val_clean
            elif 'Author' in b_text:
                rec['Author'] = val_clean
            elif 'Additional Authors' in b_text:
                rec['Additional Authors'] = val_clean
            elif 'Topics' in b_text:
                rec['Topics'] = val_clean
            elif 'Summary' in b_text:
                rec['Summary'] = val_clean
            elif 'History' in b_text:
                # History has its own container div, handle separately below
                pass

        # Also catch inline Additional Authors if formatted inside the Author line
        author_div = block_soup.find(
            string=lambda text: text and 'Author:' in text)
        if not rec.get('Author') and author_div:
            # Fallback parsing if structure varies slightly
            pass

        # 4. History List items
        history_div = block_soup.find('div', class_='historyList')
        if history_div:
            history_text = history_div.get_text(separator='\n', strip=True)
            rec['History'] = [h.strip()
                              for h in history_text.split('\n') if h.strip()]
        else:
            rec['History'] = []

        records.append(rec)

    return pd.DataFrame(records)


html_content = soup.find_all('div', id='dnn_ctr34167_StateNetDB_linkList')[0]

df = parse_statenet_html(html_content)

# Export csv to dataset
file_name = '../../data/2025-2026/ncsl_ai_db_2025_2026.csv'
df.to_csv(file_name, index=False)

# View final columns for dataset
print(df.columns)

Index(['Bill', 'Stage', 'Status', 'Date of Last Action', 'Author', 'Topics',
       'Summary', 'History'],
      dtype='str')
